# Active learning: building a sin(x) surrogate

Active learning picks the next sample where the GP posterior variance is highest. On a smooth 1-D objective, this drives the surrogate accuracy down dramatically faster than uniform random sampling.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from yee import AlConfig, active_learn

Objective: sin(x) on [0, 2π]. Pretend each query costs a full FDTD run.

In [ ]:
def f(x): return math.sin(x[0])
cfg = AlConfig(n_initial=5, n_iters=20, n_candidates=1024, seed=42)
result = active_learn(f, [(0.0, 2.0 * math.pi)], cfg)

xs_train = result.history_x[:, 0]
ys_train = result.history_y
print(f"Total evaluations: {len(ys_train)}")
print(f"First 5 (LHS) x: {xs_train[:5]}")
print(f"Next 5 AL-chosen x: {xs_train[5:10]}")

Plot the GP posterior mean ± 2σ alongside the truth.

In [ ]:
gp = result.final_gp()
xs_dense = np.linspace(0.0, 2 * math.pi, 200)
mean = np.array([gp.predict_mean(np.array([x])) for x in xs_dense])
var = np.array([gp.predict(np.array([x]))[1] for x in xs_dense])
std = np.sqrt(np.maximum(var, 0.0))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xs_dense, np.sin(xs_dense), "k--", label="true sin(x)", alpha=0.6)
ax.plot(xs_dense, mean, "C0", label="GP mean")
ax.fill_between(xs_dense, mean - 2 * std, mean + 2 * std, alpha=0.2, color="C0", label="±2σ")
ax.scatter(xs_train[:5], ys_train[:5], color="gray", marker="o",
           label="LHS initial", zorder=5)
ax.scatter(xs_train[5:], ys_train[5:], c=np.arange(len(xs_train) - 5),
           cmap="viridis", marker="x", label="AL-chosen", zorder=5)
ax.set_xlabel("x"); ax.set_ylabel("f(x)"); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title(f"GP after {len(ys_train)} AL queries")
plt.tight_layout(); plt.show()

## Convergence comparison

Now compare AL vs random sampling.

In [ ]:
from yee import GaussianProcess
rng = np.random.default_rng(42)
x_rand = rng.uniform(0.0, 2.0 * math.pi, 25).reshape(-1, 1)
y_rand = np.sin(x_rand[:, 0])
gp_rand = GaussianProcess.fit_ml(x_rand, y_rand)

x_test = np.linspace(0.0, 2 * math.pi, 100).reshape(-1, 1)
y_test = np.sin(x_test[:, 0])
rmse_al = np.sqrt(np.mean([(gp.predict_mean(x) - y)**2
                           for x, y in zip(x_test, y_test)]))
rmse_rand = np.sqrt(np.mean([(gp_rand.predict_mean(x) - y)**2
                             for x, y in zip(x_test, y_test)]))
print(f"AL RMSE:     {rmse_al:.6f}")
print(f"Random RMSE: {rmse_rand:.6f}")
print(f"AL is {rmse_rand / rmse_al:.1f}x better than random")

## Next steps — wiring AL into a real FDTD objective

Swap `f(x)` for a closure that calls `yee.FdtdDriver` on a parameterized geometry (e.g. dipole length → far-field gain at a target angle). The variance-acquisition loop will spend its compute budget where the GP is least sure — typically near sharp resonance features — rather than wasting samples on the smooth tails.

See `crates/yee-py/README.md` for the full `active_learn` / `FdtdDriver` API surface.